# Konu 7 — Görsel üretimi: yönergeden görsele

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.

Bu konuda yeni kod yazmıyoruz: hazır hücrelerde yalnızca tırnak içindeki metinleri ve sayıları değiştiriyoruz.

Isınma anahtarsız. Adım 1'den sonrası `anahtar.txt` ister (Konu 2'deki dosya).
Defterin tamamı yaklaşık 25 görsel üretir; ücretsiz günlük kotanın küçük bir kısmı.
Ürettiğin her görsel bu klasörde `ciktilar/` içine numarayla kaydedilir.

Bugünün brifi, Konu 1'deki kafe: müşterilerin **sessiz bir çalışma yeri** diye anlattığı kafe için
bir afiş görseli arıyoruz.

## Isınma — Yönergenin parçaları

İyi bir yönerge (prompt) rastgele kelimeler değil, parçaları belli bir tariftir.
Her parçayı ayrı bir değişkene yazıp f-string ile birleştiriyoruz (Konu 2'de adresi de böyle kurmuştuk).

Görsel modelleri çoğunlukla İngilizce metinle eğitildi; o yüzden yönergeyi İngilizce yazıyoruz.
Her satırın yanında Türkçesi var (`#` ile başlayan kısım yorum: Python okumaz).

In [ ]:
ozne = "a ceramic coffee cup on a wooden desk"   # özne: ahşap masada seramik bir fincan
ortam = "next to an open notebook"               # ortam: açık bir defterin yanında
stil = "watercolor illustration"                 # stil: suluboya illüstrasyon
isik = "soft morning light"                      # ışık: yumuşak sabah ışığı
kompozisyon = "top-down view"                    # kompozisyon: tepeden bakış

yonerge = f"{ozne}, {ortam}, {stil}, {isik}, {kompozisyon}"
print(yonerge)

Bir parçayı değiştirip yeniden çalıştır: `stil` satırındaki metni `"flat vector poster"` (düz vektör afiş) yap.
Yönergenin geri kalanı aynı kaldı mı? Sonra yeniden `"watercolor illustration"` yap ve hücreyi bir kez daha çalıştır.

Bugünkü bütün deneyler bu fikre dayanıyor: **bir seferde tek bir parçayı değiştir.**

## Adım 1 — İlk görsel

`gorsel_araci.py` bu klasörde duran hazır bir araç. Şimdilik onu bir arayüz gibi kullanacağız: içine yönergeyi yazıyorsun,
görsel çıkıyor. İçinde ne olduğuna (istek, anahtar, yanıtı dosyaya yazmak) **Konu 8'de** birlikte bakacağız.
Araç anahtarı Konu 2'deki gibi `../anahtar.txt` dosyasından kendisi okur.

Önce aracın üç fonksiyonunu alıyoruz (Konu 4'teki `from sklearn.decomposition import PCA` gibi):

In [ ]:
from gorsel_araci import ciz, duzenle, yan_yana

`ciz` yönergeyi bir görsel modeline gönderir, gelen görseli kaydeder ve gösterir. Kaydettiği dosyanın **yolunu** geri verir.

In [ ]:
ilk = ciz(yonerge)
print(ilk)

Görsel birkaç saniyede geldi ve `ciktilar/` klasörüne kaydedildi; `print` dosyanın yolunu yazdı.
Görsel yönergedeki her parçayı karşılıyor mu? Hangisini atlamış?

## Adım 2 — Aynı yönerge, aynı görsel mi?

Model her seferinde rastgele bir **gürültüden** (karıncalanmadan) başlar. O zaman aynı yönergeyi iki kez çalıştırınca ne olmalı?

`goster=False`: görseli hemen gösterme; `yan_yana` ikisini birlikte gösterecek.

In [ ]:
birinci = ciz(yonerge, goster=False)
ikinci = ciz(yonerge, goster=False)
yan_yana([birinci, ikinci], ["1. deneme", "2. deneme"])

Büyük ihtimalle iki farklı görsel geldi: gürültü farklı olunca görsel de farklı.

Rastgeleliği sabitlemenin yolu **tohum** (seed): gürültüyü üreten sayı. Aynı tohum, aynı gürültü demek.

In [ ]:
birinci = ciz(yonerge, tohum=42, goster=False)
ikinci = ciz(yonerge, tohum=42, goster=False)
yan_yana([birinci, ikinci], ["tohum 42", "yine tohum 42"])

Aynı model + aynı yönerge + aynı tohum = aynı görsel. Bu, deney yapabilmenin şartı: bir şeyi değiştirdiğimizde farkın
**o değişiklikten** geldiğinden emin olmak için geri kalan her şeyi sabit tutarız. Bundan sonra hep `tohum=42`.

## Adım 3 — Gürültüden görsele: adım sayısı

Görsel bir anda çıkmıyor: model gürültüden başlıyor ve her **adımda** biraz temizliyor. Bu yönteme **difüzyon** denir.
Bu modelde adım sayısını 1 ile 8 arasında seçebiliyoruz (varsayılan 4).

Aynı tohumla 1, 2, 4 ve 8 adımda üretiyoruz. Her turda görselin yolunu bir listeye ekliyoruz; sonra hepsini yan yana diziyoruz.
Dikkat: bunlar aynı görselin ara kareleri değil; her biri baştan, farklı adım sayısıyla üretiliyor.

**Tahmin et, sohbete yaz:** 1 adımda üretilen görsel, 8 adımdakinden ne kadar farklı olur?

In [ ]:
yollar = []
for adim in [1, 2, 4, 8]:
    yol = ciz(yonerge, tohum=42, adim=adim, goster=False)
    yollar.append(yol)

yan_yana(yollar, ["1 adım", "2 adım", "4 adım", "8 adım"])

1 adımla 8 adım arasındaki fark nerede: biçimde mi, renkte mi, ayrıntıda mı? Fark beklediğinden küçükse şaşırma:
bu model (FLUX.1 schnell) az adımda bitirmek için özel olarak eğitildi. Daha çok adım daha çok süre ve kota demek;
bu model 4 adımda çoğu zaman yeterince iyi.

## Adım 4 — Kontrollü deney: yalnızca stil değişsin

Tohum ve öbür dört parça (özne, ortam, ışık, kompozisyon) sabit; yalnızca **stil** değişiyor. Her turda yeni stille bir `deneme` yönergesi kuruyoruz;
Isınmadaki `yonerge` değişmeden kalıyor. `print(yol, deneme)` her görselin dosya adını ve yönergesini yazıyor: üretim günlüğüne bunları, model (`flux`) ve tohumla (`42`) birlikte yazacaksın.

**Tahmin et:** tohum aynı olduğu için üç görselin kompozisyonu birbirine benzer mi?

In [ ]:
stiller = ["watercolor illustration", "flat vector poster", "35mm film photograph"]

yollar = []
for yeni_stil in stiller:
    deneme = f"{ozne}, {ortam}, {yeni_stil}, {isik}, {kompozisyon}"
    yol = ciz(deneme, tohum=42, goster=False)
    print(yol, deneme)
    yollar.append(yol)

yan_yana(yollar, stiller, kaydet="stil_panosu.png")

`kaydet="stil_panosu.png"`: pano bu klasöre bir görsel dosyası olarak da kaydedildi.

Şimdi stil sabit, yalnızca **ışık** değişiyor. Listede üç ışık var: gece masa lambası, sert öğle güneşi,
bulutlu gün. İstersen kendi ışıklarını yaz.

In [ ]:
isiklar = ["warm desk lamp at night", "harsh midday sun", "overcast daylight"]

yollar = []
for yeni_isik in isiklar:
    deneme = f"{ozne}, {ortam}, {stil}, {yeni_isik}, {kompozisyon}"
    yol = ciz(deneme, tohum=42, goster=False)
    print(yol, deneme)
    yollar.append(yol)

yan_yana(yollar, isiklar)

Hangi ışık "sessiz bir çalışma yeri" brifine en çok uyuyor? Neden? Sohbete tek cümle yaz.

## Adım 5 — Uzun yönerge daha mı iyi?

Slaytta "iyi yönerge uzun değil, net olandır" dedik. Bu iddiayı sınıyoruz: aynı tohumla kısa ve uzun bir yönerge.
Uzun olanda birbiriyle çelişen istekler de var: hem minimal hem çok detaylı, hem gece hem güneşli, hem suluboya hem fotoğraf gerçekçi.

In [ ]:
kisa = "a ceramic coffee cup on a wooden desk, watercolor illustration, soft morning light"
uzun = "a ceramic coffee cup on a wooden desk, minimal, extremely detailed, 8k, masterpiece, at night, sunny, watercolor, photorealistic"

yol_kisa = ciz(kisa, tohum=42, goster=False)
yol_uzun = ciz(uzun, tohum=42, goster=False)
yan_yana([yol_kisa, yol_uzun], ["kısa ve net", "uzun ve çelişkili"])

Hangisi brife daha yakın? Uzun yönergedeki çelişen isteklerden model hangisini seçmiş, yoksa ikisini karıştırmış mı?
Senin denemende iddia tuttu mu? Tutmadıysa sohbete yaz; unutma, tek modelde tek tohumla yapılan tek deney kesin hüküm vermez.

## Adım 6 — Türkçe mi, İngilizce mi?

Konu 5'te Türkçenin daha çok belirtece (parçaya) bölündüğünü gördük; modeller de çoğunlukla İngilizce metinle eğitildi.
Aynı yönergeyi Türkçe ve İngilizce yazıp aynı tohumla deniyoruz.

In [ ]:
turkce = "ahşap masada seramik bir kahve fincanı, suluboya illüstrasyon, yumuşak sabah ışığı"
ingilizce = "a ceramic coffee cup on a wooden desk, watercolor illustration, soft morning light"

yol_tr = ciz(turkce, tohum=42, goster=False)
yol_en = ciz(ingilizce, tohum=42, goster=False)
yan_yana([yol_tr, yol_en], ["Türkçe", "İngilizce"])

Türkçe yönergenin hangi kısımları görsele girdi, hangileri kayboldu? Fark çıkmadıysa bu model kısa Türkçe yönergeyi iyi anlıyor
demektir; daha uzun ve ayrıntılı bir Türkçe yönergeyle dene.
Pratik kural: yönergeyi İngilizce yaz; aklına Türkçe geliyorsa önce çevir.

## Adım 7 — Başka bir model, başka bir karakter

Şimdiye kadar `flux` modelini kullandık. `model="sdxl"` başka bir firmanın modeli: Stable Diffusion XL.
Isınmadaki `yonerge` ile ve aynı tohumla ikisini yan yana koyuyoruz. (İki model farklı çalıştığı için aynı tohum aynı görseli vermez;
tohum yalnızca **aynı modelde** işe yarar.)

In [ ]:
yol_flux = ciz(yonerge, tohum=42, goster=False)
yol_sdxl = ciz(yonerge, tohum=42, model="sdxl", goster=False)
yan_yana([yol_flux, yol_sdxl], ["flux", "sdxl"])

Renk, çizgi, ayrıntı: iki modelin "eli" nerede farklı? Bir tasarımcının üslubu gibi, her modelin bir karakteri var.

`sdxl` iki şey daha alıyor:

- **Negatif yönerge** (`negatif`): görselde **istemediğin** şeyler. Bu hücrede yazı, harf, filigran ve insan.
- **En ve boy** (`en`, `boy`): görselin piksel boyutu. Yatay bir afiş için en > boy; 1024 × 576 tam 16:9.

In [ ]:
yol_afis = ciz(yonerge, tohum=42, model="sdxl", negatif="text, letters, watermark, people", en=1024, boy=576)

Görsel yatay mı geldi? Negatif yönergedeki şeylerden (yazı, filigran, insan) görselde kalan var mı? Negatif yönerge bir istek, garanti değil.

## Adım 8 — Var olanı değiştirmek: görsel düzenleme

Tasarımcının işi çoğu zaman sıfırdan üretmek değil, eldekini değiştirmek: arka planı sadeleştirmek, bir nesneyi çıkarmak,
rengi değiştirmek. `duzenle` bir görselin yolunu ve bir talimat alır; arkada üçüncü bir model (FLUX.2 klein) çalışıyor.

Adım 1'deki ilk görseli düzenliyoruz:

In [ ]:
yeni = duzenle(ilk, "make the background a plain sage green color, keep the cup the same")
yan_yana([ilk, yeni], ["önce", "sonra"])

Fincan korundu mu, yoksa model başka şeyleri de değiştirdi mi? Talimatı değiştirip dene:
`"add steam rising from the cup"` (fincandan yükselen buhar ekle), `"turn it into a pencil sketch"` (karakalem çizime çevir).

## Bonus — Kendi brifin

Seçtiğin bir marka ya da etkinlik için Isınmadaki beş parçayı yeniden yaz ve Isınma hücresini çalıştır.
Sonra Adım 4'teki stil döngüsünü kendi stillerinle çalıştır; panoyu `kaydet="odev7_pano.png"` ile kaydet. Ödev bunun üzerine kurulu.